# Matched tie-sensitivity training

Read README.md first. Use the `tie_train` kernel on the GPU server. These cells call the same scripts as the command-line runbook; training runs in fresh subprocesses. Do not change source files or environments between conditions.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

BUNDLE = Path.cwd().resolve()  # Launch Jupyter from the tie_sensitivity directory.
assert (BUNDLE / "train.py").is_file(), "Set BUNDLE to the directory containing train.py"
REPO = BUNDLE.parent.parent
ARTIFACTS = Path("/DATA/j_andrei/tie_sensitivity_artifacts")  # EDIT
def run(script, *args):
    command = [sys.executable, str(BUNDLE / script), *map(str, args)]
    print("Running:", command, flush=True)
    subprocess.run(command, check=True)


In [ ]:
SOURCE = REPO / "New folder/modernbert_supervised_baseline/modernbert_supervised_baseline/dorismae_listwise_4494.jsonl"
REVISION = os.environ.get("LLAMA_REVISION", "")  # Set to the SAME saved 40-character model commit for all runs.
assert len(REVISION) == 40, "Set REVISION to your pinned Llama commit SHA"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # EDIT if necessary, expose exactly one GPU
POLICIES = ["deterministic", "epoch_random"]  # Optionally add fixed_random
SEEDS = [42]  # After the pilot, use [2026, 3407]; do not retrain completed directories
MODE = "preflight"  # preflight -> smoke -> train (rerun the execution cell for each mode)


## Freeze supervision and check CPU logic

The preparation command copies and checks the existing artifact; it does not resample lists. Tests include a synthetic pipeline, not a real model download.


In [ ]:
run("prepare_data.py", "--source", SOURCE, "--output", ARTIFACTS / "data")
subprocess.run([sys.executable, "-B", "-m", "unittest", "discover", "-s", str(BUNDLE), "-p", "test_*.py", "-v"], check=True)


## Execute the selected phase

Start with `MODE='preflight'`, inspect both preflight manifests, then use `MODE='smoke'`. Only after the two-step GPU smoke succeeds, use `MODE='train'`. Smoke mode still checks every tokenized presentation. Full runs have two logical epochs and 4,494 optimizer steps. Each run may take substantial time; output appears as the subprocess runs.


In [ ]:
assert MODE in {"preflight", "smoke", "train"}
output = ARTIFACTS / ("smoke_runs" if MODE == "smoke" else "runs")
for seed in SEEDS:
    for policy in POLICIES:
        args = ["--data", ARTIFACTS / "data", "--output", output,
                "--policy", policy, "--seed", seed, "--revision", REVISION]
        if MODE == "preflight":
            args += ["--preflight-only"]
        elif MODE == "smoke":
            args += ["--smoke-steps", 2]
        run("train.py", *args)


## Next

Evaluate the completed pilot with notebook 02 or README steps 6–8. Then train and evaluate seeds 2026 and 3407. Changing tie targets is **not** a true tie-aware loss. Preserve run manifests and keep smoke runs out of the study.
